# 4.4 · 岭回归 / Ridge Regression (L2)

> **课程定位 / Where this fits**
> 第 4 课，**Part 4 · 监督学习：回归**。
> Lesson 4, **Part 4 · Supervised Regression**.
>
> 4.3 看到了过拟合。**正则化(regularization)** 是治过拟合最重要的武器，岭回归是它的第一个代表：在损失里加一个**惩罚系数大小**的项(L2)，逼模型"别太激进"。它用一点偏差换大量方差下降，还能治多重共线性(4.2)。
> 4.3 showed overfitting. **Regularization** is the most important cure, and Ridge is its first form: add a term to the loss that **penalizes large coefficients (L2)**, forcing the model to be less aggressive. It trades a little bias for a lot of variance reduction and cures multicollinearity (4.2).
>
> 💼 **实战/面试视角**："L1 vs L2 正则 / Ridge 怎么防过拟合 / λ 怎么选" 是 ML 面试**必考**。
> 💼 **Practical/interview angle:** "L1 vs L2 / how Ridge prevents overfitting / choosing λ" are must-ask ML questions.

> 📐 **符号约定 / Notation**
> - $\lambda$ (sklearn 里叫 `alpha`) —— 正则强度 / regularization strength
> - L2 惩罚 $=\lambda\sum_j w_j^2 = \lambda\|\mathbf{w}\|_2^2$ —— 系数平方和

> 💡 **面试相关 / Interview-relevant**
> - "Ridge 的损失函数 / L2 惩罚怎么防过拟合"（出镜率 ★★★★★）
> - "为什么 Ridge 前必须标准化"（★★★★★）
> - "λ 大小对偏差方差的影响"（★★★★★）
> - "Ridge 为什么不做特征选择（系数不为0）"（★★★★，对比 Lasso）
> - "Ridge 怎么治多重共线性"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 L2 惩罚如何"收缩"系数、为什么能防过拟合。
   Understand how the L2 penalty shrinks coefficients and prevents overfitting.
2. **从零**用正规方程实现 Ridge，对照 sklearn。
   Implement Ridge **from scratch** via the normal equation, matching sklearn.
3. 看清 Ridge 如何**治多重共线性**。
   See how Ridge cures multicollinearity.
4. 理解**系数收缩路径**与"为什么不为 0"。
   Understand the coefficient shrinkage path and why coefficients never hit 0.
5. 用 **CV / RidgeCV** 选 λ。
   Choose λ with CV / RidgeCV.

## 目录 / TOC
1. [先建直觉 + L2 惩罚 ⭐](#1)
2. [🏠 数据 + 从零实现 ⭐](#2)
3. [Ridge 治多重共线性 ⭐](#3)
4. [系数收缩路径 ⭐](#4)
5. [选 λ：CV / RidgeCV ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉 + L2 惩罚 ⭐ / Intuition & the L2 Penalty

过拟合时，模型的系数往往会变得**很大**（为了精确穿过每个训练点，曲线扭得很厉害）。正则化的想法：**在损失里加一个"系数越大、罚得越多"的项**，逼模型在"拟合好"和"系数小"之间权衡。岭回归用 **L2 惩罚**（系数平方和）：
When overfitting, coefficients tend to grow **large** (the curve wiggles hard to pass through every point). Regularization's idea: **add a term to the loss that penalizes large coefficients**, forcing a trade-off between fitting well and keeping coefficients small. Ridge uses the **L2 penalty** (sum of squared coefficients):

$$J(\mathbf{w}) = \underbrace{\frac1n\|\mathbf{Xw}-\mathbf{y}\|^2}_{\text{原 MSE 损失}} + \underbrace{\lambda\sum_j w_j^2}_{\text{L2 惩罚}}$$

- $\lambda=0$：退化成普通线性回归(OLS)。
  $\lambda=0$: reduces to ordinary least squares.
- $\lambda$ 越大：惩罚越重，系数被压得越小 → 模型越"平滑"、越简单 → **偏差↑、方差↓**。
  Larger $\lambda$: heavier penalty, smaller coefficients → smoother, simpler model → **bias↑, variance↓**.

**为什么必须标准化**(面试常问)：L2 惩罚对所有系数一视同仁地平方求和，但系数大小取决于特征量纲。不标准化的话，大量纲特征的系数天然小、被罚得轻，惩罚就不公平了。所以**用 Ridge 前必须先标准化**(3.4)。
**Why standardize is mandatory** (often asked): the L2 penalty sums squares of all coefficients equally, but coefficient size depends on feature scale. Without scaling, large-scale features get naturally small coefficients and are under-penalized — unfair. So **always standardize before Ridge** (3.4).


<a id="2"></a>
## 2. 数据 + 从零实现 ⭐ / Data & From Scratch

Ridge 也有闭式解：$\mathbf{w} = (\mathbf{X}^\top\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}$。注意那个 $+\lambda\mathbf{I}$——它给 $\mathbf{X}^\top\mathbf{X}$ 的对角加了一点，**保证矩阵可逆**（这正是它治共线性的数学原因）。一个细节：**不惩罚截距**（截距只是基准水平，不该被压小）。
Ridge also has a closed form: $\mathbf{w} = (\mathbf{X}^\top\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}$. Note the $+\lambda\mathbf{I}$ — it adds to the diagonal of $\mathbf{X}^\top\mathbf{X}$, **guaranteeing invertibility** (the mathematical reason it cures collinearity). One detail: **don't penalize the intercept** (it's just a baseline level, shouldn't be shrunk).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

data = fetch_california_housing(as_frame=True)
X, y = data.data.values, data.target.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)               # Ridge 必须标准化
Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)

def fit_ridge(X, y, lam):
    n, d = X.shape
    Xb = np.c_[np.ones(n), X]              # 加偏置列
    I = np.eye(d+1); I[0,0] = 0            # 单位阵, 但把偏置对应的对角置 0 → 不惩罚截距
    # 闭式解 w=(XᵀX+λI)⁻¹Xᵀy; +λI 保证可逆(治共线性的根源)
    w = np.linalg.solve(Xb.T @ Xb + lam*I, Xb.T @ y)
    return w[0], w[1:]                      # 截距, 系数

intercept, coefs = fit_ridge(Xtr, y_tr, lam=10.0)
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=10.0).fit(Xtr, y_tr)   # sklearn 的 alpha 就是公式里的 λ
print(f"从零 vs sklearn 系数最大差异: {np.abs(coefs - ridge.coef_).max():.6f}")
print(f"截距 intercept: 从零={intercept:.4f}, sklearn={ridge.intercept_:.4f}")
print("💡 sklearn 的 alpha = 公式里的 λ(正则强度)")


<a id="3"></a>
## 3. Ridge 治多重共线性 ⭐ / Ridge Cures Multicollinearity

4.2 说过多重共线性会让 OLS 系数**极不稳定**。Ridge 是标准解药。下面造两个几乎相同的特征 x1、x2（强共线），真实只有 x1 起作用（系数 3）。OLS 会在两者间**乱分系数**（甚至一正一负），Ridge 则倾向于把系数**平摊**到两者，稳定又合理。
4.2 noted multicollinearity makes OLS coefficients **wildly unstable**. Ridge is the standard fix. Below we make two near-identical features x1, x2 (strongly collinear) where only x1 truly matters (coefficient 3). OLS **splits the coefficient arbitrarily** (even one positive, one negative); Ridge tends to **spread it evenly**, stable and sensible.


In [ ]:
from sklearn.linear_model import LinearRegression
n = 200
x1 = rng.normal(0, 1, n)
x2 = x1 + rng.normal(0, 0.01, n)       # x2 ≈ x1, 强共线 / nearly identical to x1
y_syn = 3*x1 + rng.normal(0, 0.5, n)   # 真实只依赖 x1, 真系数 3
Xc = np.c_[x1, x2]
print("真实关系: y = 3·x1 (x2 是 x1 的近似副本)\n")
ols = LinearRegression().fit(Xc, y_syn)         # OLS: 系数在共线特征间乱分
print(f"OLS:   w1={ols.coef_[0]:+.2f}, w2={ols.coef_[1]:+.2f}  (和≈3 但各自乱分, 可能一正一负, 不稳)")
rdg = Ridge(alpha=1.0).fit(Xc, y_syn)           # Ridge: 平摊到两者
print(f"Ridge: w1={rdg.coef_[0]:+.2f}, w2={rdg.coef_[1]:+.2f}  (平摊到两者, 稳定合理)")
print("\nRidge 把系数'平摊'给共线特征 → 避免 OLS 的极端不稳定")


<a id="4"></a>
## 4. 系数收缩路径 ⭐ / Coefficient Shrinkage Path

画出系数随 λ 变化的曲线，能直观看到 Ridge 的本质：**随 λ 增大，所有系数都平滑地趋向 0，但永远不会精确等于 0**。这是 Ridge 和 Lasso 最重要的区别——**Ridge 只收缩、不剔除，所以它不做特征选择**（Lasso 4.5 会让部分系数精确变 0，实现自动特征选择）。
Plotting coefficients vs λ reveals Ridge's essence: **as λ grows, all coefficients smoothly approach 0 but never reach exactly 0.** This is the key difference from Lasso — **Ridge only shrinks, never eliminates, so it does no feature selection** (Lasso, 4.5, drives some coefficients exactly to 0 for automatic selection).


In [ ]:
alphas = np.logspace(-2, 4, 50)
# 对每个 λ 拟合 Ridge, 收集系数 → 画收缩路径 / coefficient path over λ
coef_path = np.array([Ridge(alpha=a).fit(Xtr, y_tr).coef_ for a in alphas])

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, name in enumerate(data.feature_names):
    ax.plot(alphas, coef_path[:, i], label=name)
ax.set_xscale("log"); ax.set_xlabel("λ (alpha)"); ax.set_ylabel("系数 coefficient")
ax.axhline(0, color="k", lw=0.5); ax.legend(fontsize=7, ncol=2)
ax.set_title("Ridge 系数收缩路径: λ↑ 所有系数平滑趋0(但不为0)")
plt.tight_layout(); plt.show()
print("所有系数随 λ 增大平滑收缩, 但都不精确变 0 → Ridge 不做特征选择")
print("(Lasso 4.5 会让部分系数精确=0 → 自动特征选择, 这是核心区别)")


<a id="5"></a>
## 5. 选 λ：CV / RidgeCV ⭐ / Choosing λ

λ 是要调的超参，用**交叉验证**选：太小→不够正则，仍过拟合；太大→过度正则，欠拟合；中间有个最优。`RidgeCV` 内置了高效的 CV，直接用它。
λ is a hyperparameter chosen by **cross-validation**: too small → under-regularized, still overfits; too large → over-regularized, underfits; the optimum is in between. `RidgeCV` has efficient built-in CV — use it directly.


In [ ]:
from sklearn.linear_model import RidgeCV
alphas = np.logspace(-3, 3, 50)
# 手动 CV 画 λ vs CV-R² 曲线 / CV curve over λ
cv_scores = [cross_val_score(Ridge(alpha=a), Xtr, y_tr, cv=5, scoring="r2").mean() for a in alphas]
best_alpha = alphas[np.argmax(cv_scores)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, cv_scores, "o-"); ax.axvline(best_alpha, color="r", ls="--", label=f"最优 λ={best_alpha:.3f}")
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("CV R²"); ax.legend()
ax.set_title("CV 选 λ: 太小过拟合, 太大欠拟合, 中间最优")
plt.tight_layout(); plt.show()

ridgecv = RidgeCV(alphas=alphas).fit(Xtr, y_tr)   # 自动选 λ
print(f"RidgeCV 选出 λ = {ridgecv.alpha_:.3f}")
print(f"test R²: OLS={LinearRegression().fit(Xtr,y_tr).score(Xte,y_te):.4f}, Ridge={ridgecv.score(Xte, y_te):.4f}")
print("California 特征少, Ridge 提升有限; 但高维/共线数据 Ridge 常显著胜 OLS")


<a id="6"></a>
## 6. 小结 / Summary

```
Ridge = OLS + L2 惩罚 λΣwⱼ²; 闭式解 w=(XᵀX+λI)⁻¹Xᵀy (+λI 保证可逆=治共线性)
必须标准化(L2 平等罚所有系数, 不标准化对大量纲特征不公平); 不惩罚截距
λ↑ → 系数收缩 → 偏差↑方差↓; λ=0 退化成 OLS
系数平滑趋0但不为0 → Ridge 不做特征选择(对比 Lasso 4.5)
治多重共线性: 把系数平摊给共线特征, 避免 OLS 的极端不稳定
选 λ: 交叉验证 / RidgeCV
```

### 💡 面试速查 / Interview cheat-sheet
1. **Ridge = MSE + λΣwⱼ²(L2)**；λ↑ 收缩系数, 偏差↑方差↓。
   Ridge = MSE + λΣwⱼ² (L2); larger λ shrinks coefficients, bias↑ variance↓.
2. **必须标准化**(惩罚对量纲敏感)；不惩罚截距。
   Must standardize (penalty is scale-sensitive); don't penalize the intercept.
3. **Ridge 不做特征选择**(系数趋0不为0); Lasso 才会(4.5)。
   Ridge doesn't select features (coefficients →0 not =0); Lasso does (4.5).
4. **治多重共线性**(+λI 保证可逆, 系数平摊)。
   Cures multicollinearity (+λI ensures invertibility, spreads coefficients).
5. **λ 用 CV 选**(RidgeCV)。
   Choose λ by CV (RidgeCV).

### 下一节 / Next
**4.5 Lasso (L1)**——把 L2 换成 L1 惩罚, 会把部分系数精确压到 0, 从而**自动做特征选择**。为什么 L1 能产生稀疏解是经典面试题。
**4.5 Lasso (L1)** — swap L2 for L1, which drives some coefficients exactly to 0, doing **automatic feature selection**. Why L1 yields sparsity is a classic interview question.
